# Rasterio metadata

This notebook exists because I forgot to add band names to the seasonal rasters and worldcover labels, so, instead of re-running AGAIN the entire pipeline, we will fix it here.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [44]:
from pathlib import Path 

import numpy as np
from src.io import SEASONAL_SCENES, GLOBAL_CONFIG, WORLDCOVER_LABELS
from src.config import load_config
from src.constants import  seasonal_band_names, SEASONS_ORDER
import rasterio
import rioxarray as rxr

In [12]:
cfg = load_config(GLOBAL_CONFIG)

## Seasonal features

In [13]:
SEASONAL_SCENES

[PosixPath('/Users/stepit/Repositories/projects/lulc-classification/data/processed/seasonal/DJF_SEASONAL.tif'),
 PosixPath('/Users/stepit/Repositories/projects/lulc-classification/data/processed/seasonal/MAM_SEASONAL.tif'),
 PosixPath('/Users/stepit/Repositories/projects/lulc-classification/data/processed/seasonal/JJA_SEASONAL.tif'),
 PosixPath('/Users/stepit/Repositories/projects/lulc-classification/data/processed/seasonal/SON_SEASONAL.tif')]

In [14]:
!gdalinfo {SEASONAL_SCENES[0]}

Driver: GTiff/GeoTIFF
Files: /Users/stepit/Repositories/projects/lulc-classification/data/processed/seasonal/DJF_SEASONAL.tif
       /Users/stepit/Repositories/projects/lulc-classification/data/processed/seasonal/DJF_SEASONAL.tif.aux.xml
Size is 5120, 5120
Coordinate System is:
PROJCRS["WGS 84 / UTM zone 30N",
    BASEGEOGCRS["WGS 84",
        DATUM["World Geodetic System 1984",
            ELLIPSOID["WGS 84",6378137,298.257223563,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4326]],
    CONVERSION["UTM zone 30N",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",-3,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin

We can see that `Band` does not have a field called `Description`.

In [18]:
cfg.msi.get_bands_list()

['B02_10m',
 'B03_10m',
 'B04_10m',
 'B05_20m',
 'B06_20m',
 'B07_20m',
 'B08_10m',
 'B11_20m',
 'B12_20m',
 'B8A_20m',
 'SCL_20m']

Since we would like in the future to support other data providers from the Sentinel-2 imagery, not only CDSE, we created a conversion layer on top of the names specified in the MSI config:

In [19]:
cfg.msi.band_names

{'B02_10m': 'blue',
 'B03_10m': 'green',
 'B04_10m': 'red',
 'B08_10m': 'nir',
 'B05_20m': 'red_edge1',
 'B06_20m': 'red_edge2',
 'B07_20m': 'red_edge3',
 'B8A_20m': 'narrow_nir',
 'B11_20m': 'swir1',
 'B12_20m': 'swir2',
 'SCL_20m': 'scl'}

Moreover, we don't have only reflectance bands, but also indices. So, the names we want to add to the raster are:

In [20]:
seasonal_band_names(cfg.msi)

['blue',
 'green',
 'red',
 'red_edge1',
 'red_edge2',
 'red_edge3',
 'nir',
 'swir1',
 'swir2',
 'narrow_nir',
 'NDVI',
 'NDBI',
 'NDWI']

Let's now add the information about the band names by opening the raster in read and write mode:

In [28]:
with rasterio.open(SEASONAL_SCENES[0], "r+") as src:
    src.descriptions = tuple(seasonal_band_names(cfg.msi))
    tags = src.tags()

Now we have the `Description` populated:

In [25]:
!gdalinfo {SEASONAL_SCENES[0]} | grep "Description"

  Description = blue
  Description = green
  Description = red
  Description = red_edge1
  Description = red_edge2
  Description = red_edge3
  Description = nir
  Description = swir1
  Description = swir2
  Description = narrow_nir
  Description = NDVI
  Description = NDBI
  Description = NDWI


We could also add more metadata in the form of tags, like the seaon names:

In [29]:
tags

{'AREA_OR_POINT': 'Area'}

In [32]:
for scene, season in zip(SEASONAL_SCENES, SEASONS_ORDER):
    with rasterio.open(scene, "r+") as src:
        src.descriptions = tuple(seasonal_band_names(cfg.msi))
        src.update_tags(season=season)

In [34]:
!gdalinfo {SEASONAL_SCENES[0]} | grep "season="

  season=DJF


## Labels

For the creation of the WorldCover raster, I used **rioxarray** instead of **rasterio**. But before working on the labels, we can also quickly check that the Xarray generated by opening on e of the seasonal raster with **rioxarray** contains the updated information:

In [42]:
rxr.open_rasterio(SEASONAL_SCENES[0])

<xarray.DataArray (band: 13, y: 5120, x: 5120)> Size: 1GB
[340787200 values with dtype=float32]
Coordinates:
  * band         (band) int64 104B 1 2 3 4 5 6 7 8 9 10 11 12 13
  * y            (y) float64 41kB 4.151e+06 4.151e+06 ... 4.1e+06 4.1e+06
  * x            (x) float64 41kB 2.008e+05 2.008e+05 ... 2.52e+05 2.52e+05
    spatial_ref  int64 8B 0
Attributes:
    season:                    DJF
    AREA_OR_POINT:             Area
    STATISTICS_MAXIMUM:        1.7535998821259
    STATISTICS_MEAN:           0.069649973884789
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         0.034386252708983
    STATISTICS_VALID_PERCENT:  100
    scale_factor:              1.0
    add_offset:                0.0
    long_name:                 ('blue', 'green', 'red', 'red_edge1', 'red_edg...

In [39]:
labels = rxr.open_rasterio(WORLDCOVER_LABELS)

In [43]:
labels

<xarray.DataArray (band: 1, y: 5120, x: 5120)> Size: 26MB
[26214400 values with dtype=uint8]
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 41kB 4.151e+06 4.151e+06 ... 4.1e+06 4.1e+06
  * x            (x) float64 41kB 2.008e+05 2.008e+05 ... 2.52e+05 2.52e+05
    spatial_ref  int64 8B 0
Attributes: (12/18)
    algorithm_version:  V2.0.0
    copyright:          ESA WorldCover project 2021 / Contains modified Coper...
    creation_time:      2022-10-21 07:35:40.236136
    legend:             10  Tree cover\n20  Shrubland\n30  Grassland\n40  Cro...
    license:            CC-BY 4.0 - https://creativecommons.org/licenses/by/4.0/
    product_crs:        EPSG:4326
    ...                 ...
    time_start:         2021-01-01T00:00:00Z
    title:              ESA WorldCover product at 10m resolution for year 2021
    AREA_OR_POINT:      Area
    _FillValue:         0
    scale_factor:       1.0
    add_offset:         0.0

Wait, we see that there is a lot of info already there! In particular, we can see that:

- The `product_tile` is wrong because we know from running the script for the WorldCover download that we used two tiles.
- The map from class ID to class name is present in the `legend` attributue, but is out of sync:

In [46]:
np.unique(labels)

array([1, 2, 3, 4, 5, 6, 8, 9], dtype=uint8)

So, let's remap it:

In [47]:
cfg.worldcover.class_mapping

{0: 0,
 10: 1,
 20: 2,
 30: 3,
 40: 4,
 50: 5,
 60: 6,
 70: 7,
 80: 8,
 90: 9,
 95: 10,
 100: 11}

The legend is stored as a string version of the dictionary, so we have to parse it:

In [56]:
labels.attrs["legend"]

'10  Tree cover\n20  Shrubland\n30  Grassland\n40  Cropland\n50  Built-up\n60  Bare/sparse vegetation\n70  Snow and ice\n80  Permanent water bodies\n90  Herbaceous wetland\n95  Mangroves\n100 Moss and lichen\n'

In [53]:
legend = {
    int(k): v
    for k, v in (line.split(maxsplit=1) for line in labels.attrs["legend"].strip().splitlines())
}
legend

{10: 'Tree cover',
 20: 'Shrubland',
 30: 'Grassland',
 40: 'Cropland',
 50: 'Built-up',
 60: 'Bare/sparse vegetation',
 70: 'Snow and ice',
 80: 'Permanent water bodies',
 90: 'Herbaceous wetland',
 95: 'Mangroves',
 100: 'Moss and lichen'}

In [57]:
new_legend = {}
for index, name in legend.items():
    new_legend[cfg.worldcover.class_mapping[index]] = name

new_legend

{1: 'Tree cover',
 2: 'Shrubland',
 3: 'Grassland',
 4: 'Cropland',
 5: 'Built-up',
 6: 'Bare/sparse vegetation',
 7: 'Snow and ice',
 8: 'Permanent water bodies',
 9: 'Herbaceous wetland',
 10: 'Mangroves',
 11: 'Moss and lichen'}

In [58]:
"\n".join([f"{k} {v}" for k, v in new_legend.items()])

'1 Tree cover\n2 Shrubland\n3 Grassland\n4 Cropland\n5 Built-up\n6 Bare/sparse vegetation\n7 Snow and ice\n8 Permanent water bodies\n9 Herbaceous wetland\n10 Mangroves\n11 Moss and lichen'

In [63]:
labels = labels.rio.update_attrs({"legend": "\n".join([f"{k} {v}" for k, v in new_legend.items()])})

In [65]:
labels.attrs["legend"]

'1 Tree cover\n2 Shrubland\n3 Grassland\n4 Cropland\n5 Built-up\n6 Bare/sparse vegetation\n7 Snow and ice\n8 Permanent water bodies\n9 Herbaceous wetland\n10 Mangroves\n11 Moss and lichen'

In [66]:
labels.rio.to_raster(WORLDCOVER_LABELS)